In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PowerTransformer, RobustScaler, LabelEncoder

import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('data/raw_data/Smart-Farm-IDS.csv')
print(f"Dataset shape: {df.shape}")
df.head()


Dataset shape: (172800, 16)


,Timestamp,WaterLevel,WaterPumpToTank,WaterPumpFromTank,WaterTemperature,Ec,Tds,LightIntensity,Humidity,Temperature,HeatIndex,AirQuality,SoilHumidity1,SoilHumidity2,Light,Class
0,14.6.2024 2:0:0,9,Inactive,Inactive,25.88,688.67,256,592.50,58.5,27.3,22.48,476,11.53,13.39,On,Normal
1,14.6.2024 2:0:1,9,Inactive,Inactive,25.88,688.67,256,595.00,58.6,27.3,22.48,475,11.53,13.49,On,Normal
2,14.6.2024 2:0:2,9,Inactive,Inactive,25.88,688.67,256,595.83,58.6,27.3,22.48,476,11.53,13.49,On,Normal
3,14.6.2024 2:0:3,9,Inactive,Inactive,25.88,688.67,256,595.00,58.6,27.3,22.49,475,11.34,13.59,On,Normal
4,14.6.2024 2:0:4,9,Inactive,Inactive,25.88,688.67,256,594.17,58.6,27.3,22.49,475,11.53,13.49,On,Normal


## Phase 1: Data Quality Engineering
Handling zeros and capping extreme outliers.

In [38]:
# 1. Treat suspicious zero values as missing and impute using ffill
zero_cols = ['WaterTemperature', 'Temperature', 'Ec']
for col in zero_cols:
    df[col] = df[col].replace(0, np.nan)
    
df[zero_cols] = df[zero_cols].ffill()

# 2. Cap extreme outliers for WaterTemperature and TDS
df['WaterTemperature'] = np.where(df['WaterTemperature'] > 35, 35, df['WaterTemperature'])

tds_upper = df['Tds'].quantile(0.75) + 1.5 * (df['Tds'].quantile(0.75) - df['Tds'].quantile(0.25))
df['Tds'] = np.where(df['Tds'] > tds_upper, tds_upper, df['Tds'])

# 3. Cap other outliers
heat_upper = df['HeatIndex'].quantile(0.99)
heat_lower = df['HeatIndex'].quantile(0.01)
df['HeatIndex'] = np.clip(df['HeatIndex'], heat_lower, heat_upper)

aq_upper = df['AirQuality'].quantile(0.99)
aq_lower = df['AirQuality'].quantile(0.01)
df['AirQuality'] = np.clip(df['AirQuality'], aq_lower, aq_upper)


## Phase 2: Distribution-Based Transformations
Applying log transforms for highly skewed data.

In [39]:
# Apply Log Transformation to highly positively skewed feature
df['Tds_log'] = np.log1p(df['Tds'])
df = df.drop('Tds', axis=1)

# We will let tree-based models handle moderately skewed features natively.


## Phase 3: Correlation and Redundancy Analysis
Aggregating highly correlated features.

In [40]:
# SoilHumidity1 and SoilHumidity2 are highly correlated (r = 0.958)
df['Avg_Soil_Humidity'] = (df['SoilHumidity1'] + df['SoilHumidity2']) / 2

# Drop the redundant columns to prevent multicollinearity
df = df.drop(['SoilHumidity1', 'SoilHumidity2'], axis=1)


## Phase 4: Time-Series Feature Engineering
Extracting temporal features from Timestamp.

In [41]:
# Convert Timestamp to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Extract Hour and Day of Week
df['Hour'] = df['Timestamp'].dt.hour
df['DayOfWeek'] = df['Timestamp'].dt.dayofweek

# Cyclical Encoding for Hour
df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)

# Create Night/Day indicator (e.g., Night is between 18:00 and 06:00)
df['Is_Night_Time'] = np.where((df['Hour'] < 6) | (df['Hour'] > 18), 1, 0)

# Create Weekend indicator
df['Is_Weekend'] = np.where(df['DayOfWeek'] >= 5, 1, 0)

# Drop original hour feature
df = df.drop('Hour', axis=1)
df=df.drop("Timestamp",axis=1)

## Phase 5: Domain-Specific Feature Engineering
Creating composite indicators.

In [42]:
# 1. Environmental Stress Index
df['Env_Stress'] = (df['Temperature'] * df['Humidity']) / (df['AirQuality'] + 1e-5)

# 2. Nutrient Density Ratio
df['Nutrient_Density'] = df['Tds_log'] / (df['WaterLevel'] + 1e-5)

# 3. Pump Status Mismatch (Logical Interaction)
# Need to encode WaterPumpToTank first
is_pump_active = (df['WaterPumpToTank'] == 'Active').astype(int)
is_water_high = (df['WaterLevel'] > df['WaterLevel'].quantile(0.90)).astype(int)
df['Pump_Status_Mismatch'] = is_pump_active * is_water_high


## Phases 7-9: Encoding, and Scaling
Preparing final variables for modeling.

In [43]:
# Encode Imbalanced Categoricals to Binary
binary_map = {'Inactive': 0, 'Active': 1, 'Off': 0, 'On': 1}
df['WaterPumpToTank'] = df['WaterPumpToTank'].map(binary_map).fillna(df['WaterPumpToTank'])
df['WaterPumpFromTank'] = df['WaterPumpFromTank'].map(binary_map).fillna(df['WaterPumpFromTank'])
df['Light'] = df['Light'].map(binary_map).fillna(df['Light'])

# Target Variable Label Encoding
le = LabelEncoder()
df['Class'] = le.fit_transform(df['Class']) # Normal -> 0, Attack -> 1 (verify classes)

# Set index to Timestamp for final dataset

# Scaling with RobustScaler 
features_to_scale = [c for c in df.columns if c not in ['Class', 'WaterPumpToTank', 'WaterPumpFromTank', 'Light', 'Is_Night_Time', 'Pump_Status_Mismatch']]
scaler = RobustScaler()
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

df.head()


,WaterLevel,WaterPumpToTank,WaterPumpFromTank,WaterTemperature,Ec,LightIntensity,Humidity,Temperature,HeatIndex,AirQuality,...,Tds_log,Avg_Soil_Humidity,DayOfWeek,Hour_sin,Hour_cos,Is_Night_Time,Is_Weekend,Env_Stress,Nutrient_Density,Pump_Status_Mismatch
0,0.0,0,0,-1.339286,-0.04763,-0.022492,-0.202128,-1.555556,-1.328571,0.573770,...,-0.052573,0.024148,-0.5,0.353553,0.612372,1,-0.5,-0.879893,0.007997,0
1,0.0,0,0,-1.339286,-0.04763,-0.018524,-0.191489,-1.555556,-1.328571,0.557377,...,-0.052573,0.027699,-0.5,0.353553,0.612372,1,-0.5,-0.862786,0.007997,0
2,0.0,0,0,-1.339286,-0.04763,-0.017206,-0.191489,-1.555556,-1.328571,0.573770,...,-0.052573,0.027699,-0.5,0.353553,0.612372,1,-0.5,-0.872234,0.007997,0
3,0.0,0,0,-1.339286,-0.04763,-0.018524,-0.191489,-1.555556,-1.321429,0.557377,...,-0.052573,0.024503,-0.5,0.353553,0.612372,1,-0.5,-0.862786,0.007997,0
4,0.0,0,0,-1.339286,-0.04763,-0.019841,-0.191489,-1.555556,-1.321429,0.557377,...,-0.052573,0.027699,-0.5,0.353553,0.612372,1,-0.5,-0.862786,0.007997,0


In [44]:
df.head()

,WaterLevel,WaterPumpToTank,WaterPumpFromTank,WaterTemperature,Ec,LightIntensity,Humidity,Temperature,HeatIndex,AirQuality,...,Tds_log,Avg_Soil_Humidity,DayOfWeek,Hour_sin,Hour_cos,Is_Night_Time,Is_Weekend,Env_Stress,Nutrient_Density,Pump_Status_Mismatch
0,0.0,0,0,-1.339286,-0.04763,-0.022492,-0.202128,-1.555556,-1.328571,0.573770,...,-0.052573,0.024148,-0.5,0.353553,0.612372,1,-0.5,-0.879893,0.007997,0
1,0.0,0,0,-1.339286,-0.04763,-0.018524,-0.191489,-1.555556,-1.328571,0.557377,...,-0.052573,0.027699,-0.5,0.353553,0.612372,1,-0.5,-0.862786,0.007997,0
2,0.0,0,0,-1.339286,-0.04763,-0.017206,-0.191489,-1.555556,-1.328571,0.573770,...,-0.052573,0.027699,-0.5,0.353553,0.612372,1,-0.5,-0.872234,0.007997,0
3,0.0,0,0,-1.339286,-0.04763,-0.018524,-0.191489,-1.555556,-1.321429,0.557377,...,-0.052573,0.024503,-0.5,0.353553,0.612372,1,-0.5,-0.862786,0.007997,0
4,0.0,0,0,-1.339286,-0.04763,-0.019841,-0.191489,-1.555556,-1.321429,0.557377,...,-0.052573,0.027699,-0.5,0.353553,0.612372,1,-0.5,-0.862786,0.007997,0


In [45]:
df.columns

Index(['WaterLevel', 'WaterPumpToTank', 'WaterPumpFromTank',
       'WaterTemperature', 'Ec', 'LightIntensity', 'Humidity', 'Temperature',
       'HeatIndex', 'AirQuality', 'Light', 'Class', 'Tds_log',
       'Avg_Soil_Humidity', 'DayOfWeek', 'Hour_sin', 'Hour_cos',
       'Is_Night_Time', 'Is_Weekend', 'Env_Stress', 'Nutrient_Density',
       'Pump_Status_Mismatch'],
      dtype='str')

In [46]:
df['Class'] = df.pop('Class')

## Save Processed Dataset

In [47]:
import os
os.makedirs('data/processed_data', exist_ok=True)
df.to_csv('data/processed_data/Smart-Farm-IDS_preprocessed_dataset.csv', index=False)
print("Feature Engineering completed successfully and dataset saved without the Timestamp index.")

Feature Engineering completed successfully and dataset saved without the Timestamp index.
